<a href="https://colab.research.google.com/github/25041106-rgb/POO/blob/main/UNIDAD%204/(EXAMEN_4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime
from abc import ABC, abstractmethod

# ==========================================
# 1. CLASES ABSTRACTAS
# ==========================================

class Persona(ABC):
    def __init__(self, nombre, fecha_nacimiento, correo):
        self.nombre = nombre
        self.fecha_nacimiento = fecha_nacimiento
        self.correo = correo

    @abstractmethod
    def mostrar_info(self):
        """Método abstracto: las subclases deben redefinirlo."""
        pass

class Registrable(ABC):
    @abstractmethod
    def registrar_actividad(self, accion):
        pass

# ==========================================
# 2. SUBCLASES DE PERSONA (POLIMORFISMO Y HERENCIA MÚLTIPLE)
# ==========================================

class Trabajador(Persona, Registrable):
    def __init__(self, num_trabajador, nombre, correo, fecha_nacimiento, departamento, usuario, contrasena):
        super().__init__(nombre, fecha_nacimiento, correo)
        self.num_trabajador = num_trabajador
        self.departamento = departamento
        self.usuario = usuario
        self.contrasena = contrasena
        self.actividades = []

    def mostrar_info(self):
        return f"[Trabajador] Usuario: {self.usuario}, Núm. Trabajador: {self.num_trabajador}, Nombre: {self.nombre}"

    def registrar_actividad(self, accion):
        fecha_actual = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
        registro = f"El {fecha_actual} - {self.nombre} realizó: {accion}"
        self.actividades.append(registro)

class Cliente(Persona):
    def __init__(self, num_cliente, nombre, fecha_nacimiento, correo, telefono):
        super().__init__(nombre, fecha_nacimiento, correo)
        self.num_cliente = num_cliente
        self.telefono = telefono

    def mostrar_info(self):
        return (f"[Cliente] Nombre: {self.nombre}, Correo: {self.correo}, "
                f"Núm. Cliente: {self.num_cliente}, Tel: {self.telefono}")

# ==========================================
# 3. CLASES DE DOMINIO (PRODUCTOS Y VENTAS)
# ==========================================

class Producto:
    def __init__(self, codigo, descripcion, marca, precio):
        self.codigo = codigo
        self.descripcion = descripcion
        self.marca = marca
        self.precio = precio

    def mostrar_info(self):
        print(f"[{self.codigo}] {self.descripcion} ({self.marca}) - ${self.precio}")

class InventarioProducto:
    def __init__(self, producto_obj, cantidad):
        self.producto = producto_obj
        self.total = cantidad
        self.vendidos = 0

    def disponible(self):
        return self.total - self.vendidos

    def vender(self, cantidad):
        if self.disponible() >= cantidad:
            self.vendidos += cantidad
            return True
        return False

    def mostrar_estado(self):
        p = self.producto
        print(f"[{p.codigo}] {p.descripcion} | Total: {self.total} | Disponibles: {self.disponible()} | Vendidos: {self.vendidos}")

class Venta:
    contador_ventas = 1

    def __init__(self, codigo_producto, num_cliente, fecha, cantidad, importe):
        self.folio = Venta.contador_ventas
        Venta.contador_ventas += 1
        self.codigo_producto = codigo_producto
        self.num_cliente = num_cliente
        self.fecha = fecha
        self.cantidad = cantidad
        self.importe = importe

    def mostrar_info(self):
        print(f"Folio: {self.folio} | Producto: {self.codigo_producto} | Cliente: {self.num_cliente} | Fecha: {self.fecha} | Cantidad: {self.cantidad} | Importe: ${self.importe}")

# ==========================================
# 4. OPERACIONES DEL SISTEMA
# ==========================================

def alta_producto():
    print("\n--- Alta de Producto (Ferretería) ---")
    codigo = input("Código del producto: ")
    for inv_item in inventario:
        if inv_item.producto.codigo == codigo:
            print("¡Ya existe un producto con ese código!")
            return
    descripcion = input("Descripción (Ej. Martillo Truper): ")
    marca = input("Marca: ")
    try:
        precio = float(input("Precio: "))
        cantidad = int(input("Cantidad en inventario: "))
    except ValueError:
        print("Valor inválido.")
        return

    nuevo_producto = Producto(codigo, descripcion, marca, precio)
    inventario_item = InventarioProducto(nuevo_producto, cantidad)
    inventario.append(inventario_item)
    print("Producto agregado con éxito al inventario.")


    if trabajador_activo:
        trabajador_activo.registrar_actividad(f"Alta de producto '{descripcion}'")

def alta_cliente():
    print("\n--- Alta de Cliente ---")
    num_cliente = input("Número de cliente: ")
    for c in clientes:
        if c.num_cliente == num_cliente:
            print("¡Ya existe un cliente con ese número!")
            return
    nombre = input("Nombre: ")
    fecha_nacimiento = input("Fecha de nacimiento (YYYY-MM-DD): ")
    correo = input("Correo electrónico: ")
    telefono = input("Teléfono: ")

    nuevo_cliente = Cliente(num_cliente, nombre, fecha_nacimiento, correo, telefono)
    clientes.append(nuevo_cliente)
    print("Cliente registrado exitosamente.")


    if trabajador_activo:
        trabajador_activo.registrar_actividad(f"Alta de cliente '{nombre}'")

def registrar_venta():
    print("\n--- Registrar Venta ---")
    codigo = input("Código del producto a vender: ")
    inv_item = next((item for item in inventario if item.producto.codigo == codigo), None)

    if not inv_item:
        print("Producto no encontrado en inventario.")
        return

    try:
        cantidad = int(input("Cantidad a vender: "))
    except ValueError:
        print("Cantidad inválida.")
        return

    num_cliente = input("Número de cliente (0 para público general): ")
    if num_cliente != "0":
        cliente = next((c for c in clientes if c.num_cliente == num_cliente), None)
        if not cliente:
            print("Cliente no encontrado.")
            return

    if not inv_item.vender(cantidad):
        print(f"Stock insuficiente. Disponibles: {inv_item.disponible()}")
        return

    fecha = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
    importe = inv_item.producto.precio * cantidad
    nueva_venta = Venta(codigo, num_cliente, fecha, cantidad, importe)
    ventas.append(nueva_venta)

    print(f"Venta registrada. Folio: {nueva_venta.folio} | Total a pagar: ${importe}")


    if trabajador_activo:
        trabajador_activo.registrar_actividad(f"Registró venta folio {nueva_venta.folio} por ${importe}")

# ==========================================
# 5. FUNCIONES DE APOYO Y SESIÓN
# ==========================================

def iniciar_sesion():
    print("\n=== INICIO DE SESIÓN ===")
    usuario = input("Usuario: ")
    contrasena = input("Contraseña: ")

    for t in trabajadores:
        if t.usuario == usuario and t.contrasena == contrasena:
            print(f"Bienvenido, {t.nombre} ({t.departamento})")
            t.registrar_actividad("Inicio de sesión")
            return t
    print("Usuario o contraseña incorrectos.\n")
    return None

def cerrar_sesion():
    global trabajador_activo
    if trabajador_activo:
        trabajador_activo.registrar_actividad(f"Cierre de sesión")
        print(f"Sesión cerrada, {trabajador_activo.nombre}.")
    trabajador_activo = None

def mostrar_registro_actividad():
    print("\n--- Registro de Actividad en el Sistema ---")
    if trabajador_activo and trabajador_activo.actividades:
        for act in trabajador_activo.actividades:
            print("*", act)
    else:
        print("No hay actividades registradas para el trabajador activo.")

# ==========================================
# 6.  EJECUCIÓN
# ==========================================

def mostrar_menu():
    print("\n--- SISTEMA DE GESTIÓN FERRETERÍA ---")
    print("1. Alta de producto")
    print("2. Alta de cliente")
    print("3. Registrar venta")
    print("4. Mostrar inventario")
    print("5. Mostrar historial de ventas")
    print("6. Mostrar todos los clientes")
    print("7. Mostrar registro de mi actividad")
    print("8. Cerrar sesión")
    print("9. Salir del sistema")

def iniciar_sistema():
    global trabajador_activo
    trabajador_activo = None

    while True:
        if trabajador_activo is None:
            trabajador_activo = iniciar_sesion()
            if trabajador_activo is None:
                continue

        mostrar_menu()
        opcion = input("Seleccione una opción: ")

        if opcion == "1":
            alta_producto()
        elif opcion == "2":
            alta_cliente()
        elif opcion == "3":
            registrar_venta()
        elif opcion == "4":
            print("\n--- Inventario de la Ferretería ---")
            for item in inventario:
                item.mostrar_estado()
        elif opcion == "5":
            print("\n--- Historial de Ventas ---")
            for v in ventas:
                v.mostrar_info()
        elif opcion == "6":
            print("\n--- Lista de Clientes ---")
            for c in clientes:
                print(c.mostrar_info())
        elif opcion == "7":
            mostrar_registro_actividad()
        elif opcion == "8":
            cerrar_sesion()
        elif opcion == "9":
            print("Saliendo del sistema...")
            break
        else:
            print("Opción no válida.")


inventario = []
clientes = []
ventas = []
trabajadores = []
trabajador_activo = None


admin = Trabajador("0001", "Administrador", "admin@ferreteria.com", "1980-01-01", "Gerencia", "admin", "admin")
trabajadores.append(admin)

if __name__ == "__main__":
    iniciar_sistema()



=== INICIO DE SESIÓN ===
Usuario: admin
Contraseña: admin
Bienvenido, Administrador (Gerencia)

--- SISTEMA DE GESTIÓN FERRETERÍA ---
1. Alta de producto
2. Alta de cliente
3. Registrar venta
4. Mostrar inventario
5. Mostrar historial de ventas
6. Mostrar todos los clientes
7. Mostrar registro de mi actividad
8. Cerrar sesión
9. Salir del sistema
Seleccione una opción: 1

--- Alta de Producto (Ferretería) ---
Código del producto: 1234
Descripción (Ej. Martillo Truper): DESARMADORES
Marca: stilson
Precio: 299
Cantidad en inventario: 20
Producto agregado con éxito al inventario.

--- SISTEMA DE GESTIÓN FERRETERÍA ---
1. Alta de producto
2. Alta de cliente
3. Registrar venta
4. Mostrar inventario
5. Mostrar historial de ventas
6. Mostrar todos los clientes
7. Mostrar registro de mi actividad
8. Cerrar sesión
9. Salir del sistema
Seleccione una opción: 2

--- Alta de Cliente ---
Número de cliente: 12
Nombre: jasiel
Fecha de nacimiento (YYYY-MM-DD): 20-07-2006
Correo electrónico: jasiel@gm